pip install pandas

# Load files in

In [150]:
import pandas as pd

df_lib_book = pd.read_csv('./data/03_LibrarySystemBook.csv')
df_lib_customers = pd.read_csv('./data/03_LibrarySystemCustomers.csv')

In [151]:
print("First 5 rows of LibrarySystemBook")
df_lib_book.head()

First 5 rows of LibrarySystemBook


,Id,Books,Book checkout,Book Returned,Days allowed to borrow,Customer ID
0,1.0,Catcher in the Rye,"""20/02/2023""",25/02/2023,2 weeks,1.0
1,2.0,Lord of the rings the two towers,"""24/03/2023""",21/03/2023,2 weeks,2.0
2,3.0,Lord of the rings the return of the kind,"""29/03/2023""",25/03/2023,2 weeks,3.0
3,4.0,The hobbit,"""02/04/2023""",25/03/2023,2 weeks,4.0
4,5.0,Dune,"""02/04/2023""",25/03/2023,2 weeks,5.0


In [152]:
print("Info of LibrarySystemBook")
df_lib_book.info()

Info of LibrarySystemBook
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 114 entries, 0 to 113
Data columns (total 6 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Id                      21 non-null     float64
 1   Books                   20 non-null     object 
 2   Book checkout           21 non-null     object 
 3   Book Returned           21 non-null     object 
 4   Days allowed to borrow  21 non-null     object 
 5   Customer ID             20 non-null     float64
dtypes: float64(2), object(4)
memory usage: 5.5+ KB


# Clean the files

In [153]:
# First dataset - LibrarySystemBook

df_lib_book_clean = df_lib_book.copy()

# Drop any empty rows
df_lib_book_clean = df_lib_book_clean.dropna(how='all')

# Clean Ids in Lib Book DF, use index as Id where NA/Null value
df_lib_book_clean = df_lib_book_clean.rename(columns={'Id': 'Book ID'})
df_lib_book_clean['Book ID'] = df_lib_book_clean['Book ID'].fillna(0).astype(int)
df_lib_book_clean['Book ID'] = df_lib_book_clean.index + 1

# Clean and format Books column
df_lib_book_clean['Books'] = df_lib_book_clean['Books'].fillna('Missing Book Title').astype(str)
df_lib_book_clean['Books'] = df_lib_book_clean['Books'].str.title()

# Clean and format Book Checkout column
df_lib_book_clean = df_lib_book_clean.rename(columns={'Book checkout': 'Checkout'})
# Used Regex here rather than strip to strip anything non digit or date separator
df_lib_book_clean['Checkout'] = df_lib_book_clean['Checkout'].str.replace(r'[^0-9/\-\.]', '', regex=True)
df_lib_book_clean['Checkout'] = pd.to_datetime(df_lib_book_clean['Checkout'], errors='coerce', dayfirst=True)

# Clean and format Book Returned column
df_lib_book_clean = df_lib_book_clean.rename(columns={'Book Returned': 'Returned'})
df_lib_book_clean['Returned'] = df_lib_book_clean['Returned'].str.replace(r'[^0-9/\-\.]', '', regex=True)
df_lib_book_clean['Returned'] = pd.to_datetime(df_lib_book_clean['Returned'], errors='coerce', dayfirst=True)

# Impute any empty date values
df_lib_book_clean['Checkout'] = df_lib_book_clean['Checkout'].fillna(df_lib_book_clean['Returned'])
df_lib_book_clean['Returned'] = df_lib_book_clean['Returned'].fillna(df_lib_book_clean['Checkout'])

# Clean and format days allowed to borrow column
df_lib_book_clean = df_lib_book_clean.rename(columns={'Days allowed to borrow': 'Borrow Allowance (Weeks)'})
df_lib_book_clean['Borrow Allowance (Weeks)'] = df_lib_book_clean['Borrow Allowance (Weeks)'].str.replace(r'[^0-9]', '', regex=True)
df_lib_book_clean['Borrow Allowance (Weeks)'] = df_lib_book_clean['Borrow Allowance (Weeks)'].str.strip()
df_lib_book_clean['Borrow Allowance (Weeks)'] = df_lib_book_clean['Borrow Allowance (Weeks)'].fillna(0).astype(int)

# Clean Customer Ids in Lib Book DF, use index as Id where NA/Null value
df_lib_book_clean['Customer ID'] = df_lib_book_clean['Customer ID'].fillna(0).astype(int)

df_lib_book_clean.info()

<class 'pandas.core.frame.DataFrame'>
Index: 21 entries, 0 to 20
Data columns (total 6 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   Book ID                   21 non-null     int64         
 1   Books                     21 non-null     object        
 2   Checkout                  21 non-null     datetime64[ns]
 3   Returned                  21 non-null     datetime64[ns]
 4   Borrow Allowance (Weeks)  21 non-null     int32         
 5   Customer ID               21 non-null     int32         
dtypes: datetime64[ns](2), int32(2), int64(1), object(1)
memory usage: 1008.0+ bytes


In [154]:
# Second dataset - LibrarySystemCustomers

df_lib_customers_clean = df_lib_customers.copy()

# Drop any empty rows
df_lib_customers_clean = df_lib_customers_clean.dropna(how='all')

# Clean Ids in Lib Customers DF
df_lib_customers_clean['Customer ID'] = df_lib_customers_clean['Customer ID'].fillna(0).astype(int)

df_lib_customers_clean.head()

,Customer ID,Customer Name
0,1,Jane Doe
1,2,John Smith
2,3,Dan Reeves
4,5,William Holden
5,6,Jaztyn Forest


# Output the files


In [155]:
df_lib_book_clean.to_csv('./data/cleaned_LibrarySystemBook.csv', index=False)
df_lib_customers_clean.to_csv('./data/cleaned_LibrarySystemCustomers.csv', index=False)

# Add some data engineering metrics to track

In [156]:
# Track how many books a customer has checked out
books_per_customer = df_lib_book_clean.groupby('Customer ID')['Book ID'].count().reset_index()
books_per_customer.columns = ['Customer ID', 'Total Books']

df_lib_customers_clean = df_lib_customers_clean.merge(books_per_customer, on='Customer ID', how='left')
df_lib_customers_clean['Total Books'] = df_lib_customers_clean['Total Books'].fillna(0).astype(int)

# Track how many days a customer has a book for on average
df_lib_book_clean['Checkout Duration (Days)'] = (df_lib_book_clean['Returned'] - df_lib_book_clean['Checkout']).dt.days

avg_checkout = df_lib_book_clean.groupby('Customer ID')['Checkout Duration (Days)'].mean().reset_index()
avg_checkout.columns = ['Customer ID', 'Avg Checkout Days']
avg_checkout['Avg Checkout Days'].round(1)

# Join back to customers dataframe
df_lib_customers_clean = df_lib_customers_clean.merge(avg_checkout, on='Customer ID', how='left')


df_lib_customers_clean.head()

# Output new file
df_lib_customers_clean.to_csv('./data/cleaned_LibrarySystemCustomers.csv', index=False)

# Create a new dataset to show erroneous information